## MRI-Safe RandAugment

This section applies an MRI-safe RandAugment strategy to generate additional training images and reduce class imbalance. The implementation uses conservative grayscale-preserving transformations, including rotation, translation, shear, contrast, brightness, and sharpness adjustments.

Following the approach described in the HayabusaNet study, augmentation is applied only to the training set, while the validation set remains unchanged. The operation selection is performed without replacement to avoid redundant transformation combinations.

**References**

- Cubuk, E. D., Zoph, B., Shlens, J., & Le, Q. V. (2020). *RandAugment: Practical automated data augmentation with a reduced search space*. CVPR Workshops.
- Prayogo, R. D., Karimah, S. A., & Nambo, H. (2026). *HayabusaNet: Hybrid attention-based multiscale fusion CNN for accurate and efficient brain tumor classification in MRI scans*. Expert Systems with Applications, 331, 133135.

In [ ]:
import os
import random
import hashlib
from collections import defaultdict
from PIL import Image, ImageEnhance
import numpy as np

# This notebook assumes that the dataset has already been preprocessed and organized into:
SOURCE_DIR = "dataset/train_original"
TARGET_DIR = "outputs/randaugment"

TARGET_TOTAL_PER_CLASS = 2500  # Set None to balance to the largest existing class.
N_MIN, N_MAX = 2, 4
M_MAG = 6

RANDOM_SEED = 42
SAVE_PNG = True
OVERWRITE = False
VALID_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".tif")

random.seed(RANDOM_SEED)
os.makedirs(TARGET_DIR, exist_ok=True)


def _clamp01(x):
    return max(0.0, min(1.0, float(x)))


def _sample_signed(max_abs):
    return random.uniform(-max_abs, max_abs)


def _level_to_range(M, lo, hi, signed=False):
    alpha = _clamp01(M / 10.0)
    val = lo + alpha * (hi - lo)
    return _sample_signed(val) if signed else val


def estimate_bg_gray(imgL):
    """Estimate background intensity from image borders."""
    a = np.array(imgL)
    border = np.concatenate([a[0, :], a[-1, :], a[:, 0], a[:, -1]], axis=0)
    return int(np.median(border))


def rotate_mri_with_fill(imgL, deg):
    bg = estimate_bg_gray(imgL)
    return imgL.rotate(deg, resample=Image.BILINEAR, fillcolor=bg)


def affine_mri_with_fill(imgL, matrix):
    bg = estimate_bg_gray(imgL)
    return imgL.transform(
        imgL.size,
        Image.AFFINE,
        matrix,
        resample=Image.BILINEAR,
        fillcolor=bg
    )


def op_identity(imgL, M):
    return imgL


def op_rotate_mri(imgL, M):
    deg = _level_to_range(M, 0, 8, signed=True)
    return rotate_mri_with_fill(imgL, deg)


def op_translate_x_mri(imgL, M):
    px = _level_to_range(M, 0.0, 0.08, signed=True) * imgL.size[0]
    return affine_mri_with_fill(imgL, (1, 0, px, 0, 1, 0))


def op_translate_y_mri(imgL, M):
    py = _level_to_range(M, 0.0, 0.08, signed=True) * imgL.size[1]
    return affine_mri_with_fill(imgL, (1, 0, 0, 0, 1, py))


def op_shear_x_mri(imgL, M):
    sh = _level_to_range(M, 0.0, 0.04, signed=True)
    return affine_mri_with_fill(imgL, (1, sh, 0, 0, 1, 0))


def op_shear_y_mri(imgL, M):
    sh = _level_to_range(M, 0.0, 0.04, signed=True)
    return affine_mri_with_fill(imgL, (1, 0, 0, sh, 1, 0))


def op_contrast_mri(imgL, M):
    fac = _level_to_range(M, 0.95, 1.05)
    return ImageEnhance.Contrast(imgL).enhance(fac)


def op_brightness_mri(imgL, M):
    fac = _level_to_range(M, 0.95, 1.05)
    return ImageEnhance.Brightness(imgL).enhance(fac)


def op_sharpness_mri(imgL, M):
    fac = _level_to_range(M, 0.9, 1.1)
    return ImageEnhance.Sharpness(imgL).enhance(fac)


MRI_SAFE_OPS = [
    ("Identity", op_identity),
    ("Rotate", op_rotate_mri),
    ("TranslateX", op_translate_x_mri),
    ("TranslateY", op_translate_y_mri),
    ("ShearX", op_shear_x_mri),
    ("ShearY", op_shear_y_mri),
    ("Contrast", op_contrast_mri),
    ("Brightness", op_brightness_mri),
    ("Sharpness", op_sharpness_mri),
]


def rand_augment(imgL, M=M_MAG, n_min=N_MIN, n_max=N_MAX):
    """Apply MRI-safe RandAugment operations to one grayscale image."""
    pool = MRI_SAFE_OPS
    N = random.randint(n_min, n_max)
    N = max(1, min(N, len(pool)))

    ops = random.sample(pool, k=N)

    out = imgL
    for _, fn in ops:
        out = fn(out, M)

    return out


def is_image_file(name):
    return name.lower().endswith(VALID_EXTS)


def rel_parts(path, base):
    rel_path = os.path.relpath(path, base)
    rel_dir = os.path.dirname(rel_path)
    parts = rel_dir.split(os.sep) if rel_dir else []

    cls = parts[0] if len(parts) >= 1 else None
    view = parts[1] if len(parts) >= 2 else None
    base_noext, _ = os.path.splitext(os.path.basename(path))

    return cls, view, rel_dir, base_noext


def out_dir_for_rel_dir(rel_dir):
    out_dir = os.path.join(TARGET_DIR, rel_dir)
    os.makedirs(out_dir, exist_ok=True)
    return out_dir


def md5_8(s):
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:8]


def collect_files_by_group(source_dir):
    files_by_group = defaultdict(list)

    for root, _, files in os.walk(source_dir):
        for filename in files:
            if not is_image_file(filename):
                continue

            src_path = os.path.join(root, filename)
            cls, view, _, _ = rel_parts(src_path, source_dir)

            if cls and view:
                files_by_group[(cls, view)].append(src_path)

    return files_by_group


def compute_targets_by_class_view(files_by_group, target_total_per_class=None):
    class_view_counts = defaultdict(dict)

    for (cls, view), files in files_by_group.items():
        class_view_counts[cls][view] = len(files)

    class_totals = {
        cls: sum(view_counts.values())
        for cls, view_counts in class_view_counts.items()
    }

    if not class_totals:
        raise ValueError(f"No source images found in: {SOURCE_DIR}")

    if target_total_per_class is None:
        target_total_per_class = max(class_totals.values())

    targets = {}

    for cls, view_counts in class_view_counts.items():
        original_total = sum(view_counts.values())
        augment_total = max(0, target_total_per_class - original_total)

        targets[cls] = {}

        if augment_total == 0:
            for view in view_counts:
                targets[cls][view] = 0
            continue

        total_views = sum(view_counts.values())
        raw_alloc = {
            view: augment_total * count / total_views
            for view, count in view_counts.items()
        }

        base_alloc = {
            view: int(np.floor(value))
            for view, value in raw_alloc.items()
        }

        remainder = augment_total - sum(base_alloc.values())
        ranked_views = sorted(
            raw_alloc,
            key=lambda v: (raw_alloc[v] - base_alloc[v], view_counts[v]),
            reverse=True
        )

        for view in ranked_views[:remainder]:
            base_alloc[view] += 1

        targets[cls] = base_alloc

    return targets, class_totals, target_total_per_class


def process_per_class_view(
    source_dir=SOURCE_DIR,
    target_total_per_class=TARGET_TOTAL_PER_CLASS,
    save_png=SAVE_PNG,
    overwrite=OVERWRITE
):
    files_by_group = collect_files_by_group(source_dir)
    targets, class_totals, final_target = compute_targets_by_class_view(
        files_by_group,
        target_total_per_class
    )

    print("Original class distribution:")
    for cls, total in sorted(class_totals.items()):
        print(f" - {cls}: {total}")

    print(f"\nTarget total per class: {final_target}\n")

    for cls, view_map in targets.items():
        for view, target_n in view_map.items():
            if target_n <= 0:
                print(f"Skip {cls}/{view} (target=0).")
                continue

            src_list = files_by_group.get((cls, view), [])
            if not src_list:
                print(f"Warning: no source images found for {cls}/{view}.")
                continue

            print(f"Generating {cls}/{view}: {target_n} images...")
            generated, idx, attempts = 0, 0, 0
            max_attempts = target_n * max(len(src_list), 1) * 10

            while generated < target_n and attempts < max_attempts:
                src_path = src_list[idx % len(src_list)]
                idx += 1
                attempts += 1

                try:
                    imgL = Image.open(src_path).convert("L")
                except Exception as e:
                    print(f"Skip: {src_path} | {e}")
                    continue

                _, _, rel_dir, base = rel_parts(src_path, source_dir)
                out_dir = out_dir_for_rel_dir(rel_dir)

                tag = md5_8(f"{src_path}|{idx}|{generated}|{RANDOM_SEED}")
                ext = "png" if save_png else "jpg"
                out_name = f"{base}_RA{N_MIN}-{N_MAX}M{M_MAG}_{tag}.{ext}"
                out_path = os.path.join(out_dir, out_name)

                if (not overwrite) and os.path.exists(out_path):
                    continue

                augL = rand_augment(imgL, M=M_MAG, n_min=N_MIN, n_max=N_MAX)

                try:
                    if save_png:
                        augL.save(out_path, format="PNG")
                    else:
                        augL.save(out_path)

                    generated += 1
                except Exception as e:
                    print(f"Save failed: {out_path} | {e}")

            print(
                f"{cls}/{view}: generated {generated}/{target_n} images "
                f"-> {os.path.join(TARGET_DIR, cls, view)}"
            )

            if generated < target_n:
                print(
                    f"Warning: target not fully reached for {cls}/{view}. "
                    f"Try setting OVERWRITE=True or clearing the output folder."
                )


if __name__ == "__main__":
    process_per_class_view()